## Power Iteration - CuPy - Asynchrony

### Table of Contents
1. [Introduction and Setup](#1-Introduction-and-Setup)
2. [Theory: Streams and Synchronization](#2-Theory:-Streams-and-Synchronization)
3. [The Baseline Implementation](#3-The-Baseline-Implementation)
4. [Profiling the Baseline](#4-Profiling-the-Baseline)
5. [Better Visibility with NVTX](#5-Better-Visibility-with-NVTX)
6. [Implementing Asynchrony](#6-Implementing-Asynchrony)
7. [Performance Analysis](#7-Performance-Analysis)
8. [Balancing CPU I/O and GPU Compute](#8-Balancing-CPU-I/O-and-GPU-Compute)

### 1. Introduction and Setup

GPU programming is inherently asynchronous. In this exercise, we will explore the implications of this behavior when using CuPy and learn how to analyze the flow of execution using profiling tools.

We will revisit the Power Iteration algorithm. Our goal is to take a standard implementation, profile it to identify bottlenecks caused by implicit synchronization, and then optimize it using CUDA streams and asynchronous memory transfers.

First, we need to ensure the Nsight Systems profiler (nsys), Nsightful, and NVTX are installed and available.

In [ ]:
import os

# Install necessary packages if running in Google Colab.
if os.getenv("COLAB_RELEASE_TAG") and not os.path.exists("/accelerated-computing-hub-installed"):
  print("Downloading Nsight Systems package.")
  !curl -s -L -O https://developer.nvidia.com/downloads/assets/tools/secure/nsight-systems/2026_1/NsightSystems-linux-cli-public-2026.1.1.204-3717666.deb
  print("Installing Nsight Systems package.")
  !dpkg -i NsightSystems-linux-cli-public-2026.1.1.204-3717666.deb > /dev/null
  print("Installing PIP packages.")
  !pip install "nvtx" "nsightful[notebook] @ git+https://github.com/brycelelbach/nsightful.git@196bc4e292571973969179d7d48e687a2720bb2f" > /dev/null 2>&1
  open("/accelerated-computing-hub-installed", "a").close()
  print("All packages installed.")

import numpy as np
import cupy as cp
import cupyx as cpx
import nvtx
import time
from dataclasses import dataclass
from jupyter_dark_detect import is_dark
import matplotlib.pyplot as plt
plt.style.use('dark_background' if is_dark() else 'default')


### 2. Theory: Streams and Synchronization

All GPU work is launched asynchronously on a stream. The work items in a stream are executed in order. If you launch `f` on a stream and later launch `g` on that same stream, then `f` will be executed before `g`. But if `f` and `g` are launched on different streams, then their execution might overlap.

**How CuPy handles this:**

- **Default Stream:** Unless specified, CuPy launches work on the default CUDA stream.
- **Sequential Device Execution:** By default, CuPy work executes sequentially on the GPU.
- **Asynchronous Host Execution:** From the Python (Host) perspective, the code often returns immediately after launching the GPU kernel, before the work is actually finished.

**TODO:** Even though CuPy is asynchronous, certain operations force the CPU to wait for the GPU to finish. What operations do you think implicitly synchronize the host and device?

### 3. The Baseline Implementation

We will start with a baseline implementation of the Power Iteration algorithm.

The setup below mirrors the previous memory-spaces notebook: one configuration, one generated matrix, and one estimator function. The Nsight Systems kernel lets us profile these ordinary notebook cells directly.

In [ ]:
@dataclass
class PowerIterationConfig:
  dim: int = 19000
  dominance: float = 0.05
  max_steps: int = 400
  check_frequency: int = 25
  progress: bool = True
  residual_threshold: float = 1e-10

def generate_device(cfg=PowerIterationConfig()):
  cp.random.seed(42)
  weak_lam = cp.random.random(cfg.dim - 1) * (1.0 - cfg.dominance)
  lam = cp.random.permutation(cp.concatenate((cp.asarray([1.0]), weak_lam)))
  P = cp.random.random((cfg.dim, cfg.dim))
  D = cp.diag(cp.random.permutation(lam))
  return (P @ D) @ cp.linalg.inv(P)

def estimate_device_baseline(A, cfg=PowerIterationConfig()) -> np.ndarray:
  A_gpu = cp.asarray(A)
  x = cp.ones(A_gpu.shape[0], dtype=np.float64)

  for i in range(0, cfg.max_steps, cfg.check_frequency):
    y = A_gpu @ x
    lam = (x @ y) / (x @ x)
    res = cp.linalg.norm(y - lam * x)
    x = y / cp.linalg.norm(y)

    if cfg.progress:
      print(f"step {i}: residual = {res:.3e}")

    np.savetxt(f"/tmp/device_{i}.txt", cp.asnumpy(x))
    if res < cfg.residual_threshold:
      break

    for _ in range(i + 1, min(i + cfg.check_frequency, cfg.max_steps)):
      y = A_gpu @ x
      x = y / cp.linalg.norm(y)

  return cp.asnumpy((x.T @ (A_gpu @ x)) / (x.T @ x))

A_device = generate_device()
estimate_device_baseline(
  A_device,
  cfg=PowerIterationConfig(max_steps=1, check_frequency=1, progress=False),
)


### 4. Profiling the Baseline

Select the **Python 3 (Nsight Systems)** kernel, then profile the baseline in place. The `%%nsys` magic collects only this cell and displays the resulting timeline without restarting the kernel.

In [ ]:
%%nsys -o power_iteration__baseline.nsys-rep
lam_est_baseline = estimate_device_baseline(A_device)
np.testing.assert_allclose(lam_est_baseline, 1, atol=1e-4)


The timeline is displayed below the profiled cell. Explore what's going on in the program.

**EXTRA CREDIT:** Download the Nsight Systems GUI and open the report in it to see even more information.

In [ ]:
# The native report is saved as power_iteration__baseline.nsys-rep.

### 5. Better Visibility with NVTX

Nsight Systems shows us a lot of information - sometimes it's too much and not all relevant.

There's two ways that we can filter and annotate what we see in Nsight systems.

The first is to limit when we start and stop profiling in the program. In Python, we can do this with `cupyx.profiler.profile()`, which give us a Python context manager. Any CUDA code used during scope will be included in the profile.

```
not_in_the_profile()
with cpx.profiler.profile():
  in_the_profile()
not_in_the_profile()
```

For this to work, we have to pass `--capture-range=cudaProfilerApi --capture-range-end=stop` as flags to `nsys`.

We can also annotate specific regions of our code, which will show up in the profiler. We can even add categories, domains, and colors to these regions, and they can be nested. To add these annotations, we use `nvtx.annotate()`, another Python context manager, this time from a library called NVTX.

```
with nvtx.annotate("Loop"):
  for i in range(20):
     with nvtx.annotate(f"Step {i}"):
       pass
```

**TODO:** Go back to the earlier cells and improve the profile results by adding:

- `nvtx.annotate()` regions. Remember, you can nest them.
- A `cpx.profiler.profile()` around the `start =`/`stop =` lines that run the solver.
- `--capture-range=cudaProfilerApi --capture-range-end=stop` to the `nsys` flags.

Then, capture another profile and see if you can identify how we can improve the code. Specifically, think about how we could add more asynchrony.

### 6. Implementing Asynchrony

Remember what we've learned about streams and how to use them with CuPy:

- By default, all CuPy operations within a single thread run on the same stream. You can access this stream with `cp.cuda.get_current_stream()`.
- You can create a new stream with `cp.cuda.Stream(non_blocking=True)`. Use `with` statements to use the stream for all CuPy operations within a block.
- You can record an event on a stream by calling `.record()` on it.
- You can synchronize on an event (or an entire stream) by calling `.synchronize()` on it.
- Memory transfers will block by default. You can launch them asynchronously with `cp.asarray(..., blocking=False)` (for host to device transfers) and `cp.asnumpy(..., blocking=False)` (for device to host transfers).

**TODO:** Adapt the baseline estimator in the cell below to improve performance by overlapping checkpoint copies and CPU I/O with GPU compute.

In [ ]:
def estimate_device_async(A, cfg=PowerIterationConfig()) -> np.ndarray:
  raise NotImplementedError("TODO: Adapt the baseline estimator to overlap checkpoint copies and I/O with GPU compute!")


Now let's make sure it works:

In [ ]:
lam_est_async = estimate_device_async(A_device)
np.testing.assert_allclose(lam_est_async, 1, atol=1e-4)


### 7. Performance Analysis

Before we profile the improved code, let's compare the execution times of both.

In [ ]:
def time_estimator(estimator, cfg):
  cp.cuda.get_current_stream().synchronize()
  start = time.perf_counter()
  result = estimator(A_device, cfg=cfg)
  cp.cuda.get_current_stream().synchronize()
  return result, (time.perf_counter() - start) * 1000

quiet_cfg = PowerIterationConfig(progress=False)
_, power_iteration_baseline_duration = time_estimator(estimate_device_baseline, quiet_cfg)
_, power_iteration_async_duration = time_estimator(estimate_device_async, quiet_cfg)
speedup = power_iteration_baseline_duration / power_iteration_async_duration

print(f"power_iteration_baseline: {power_iteration_baseline_duration:.3f} ms")
print(f"power_iteration_async:    {power_iteration_async_duration:.3f} ms")
print(f"power_iteration_async speedup over power_iteration_baseline: {speedup:.2f}")


Next, let's capture a profile report of our improved code with the same Nsight Systems kernel.

In [ ]:
%%nsys -o power_iteration__async.nsys-rep
lam_est_async = estimate_device_async(A_device)
np.testing.assert_allclose(lam_est_async, 1, atol=1e-4)


Finally, let's look at the profile in Perfetto and confirm we've gotten rid of the idling.

In [ ]:
# The native report is saved as power_iteration__async.nsys-rep.

### 8. Balancing CPU I/O and GPU Compute

Our asynchronous implementation overlaps checkpoint I/O on the CPU with power-iteration steps on the GPU. `check_frequency` controls how many GPU steps run between residual checks and checkpoints. A smaller value produces output more frequently, but may not provide enough GPU work to hide the I/O. A larger value provides more GPU work to overlap with the I/O, but delays output and convergence checks.

**TODO:** Sweep check frequencies from 20 through 35 in increments of 1 and determine the output frequency with the lowest execution time. Use `cupyx.profiler.benchmark` with progress disabled, then plot the results.

In [ ]:
check_frequencies = list(range(20, 36))
async_durations = []

for check_frequency in check_frequencies:
    cfg = PowerIterationConfig(check_frequency=check_frequency, progress=False)
    timing = cpx.profiler.benchmark(
        estimate_device_async, (A_device, cfg), n_repeat=5, n_warmup=1
    )
    async_durations.append(timing.cpu_times.mean() * 1000)


In [ ]:
# TODO: Find the check frequency with the shortest execution time and
# display a performance graph. Highlight the optimal point.

Finally, profile the optimal check frequency. In the trace, compare the CPU I/O region with the GPU compute between checks. Does the GPU compute fully hide the I/O?

In [ ]:
%%nsys -o power_iteration__async__optimal.nsys-rep
optimal_cfg = PowerIterationConfig(check_frequency=optimal_check_frequency)
lam_est_optimal = estimate_device_async(A_device, cfg=optimal_cfg)
np.testing.assert_allclose(lam_est_optimal, 1, atol=1e-4)
